In [ ]:
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

np.random.seed(42)
print("All imports successful.")

All imports successful.


---
## Task 1: Trained Machine Learning Model


In [5]:
# Load and preprocess data
df = pd.read_csv("data-a2 (1).csv")

# Drop addref (unique ID, not a feature) and assembly (69% missing)
df = df.drop(columns=["addref", "assembly"], errors="ignore")

# Drop rows with missing price
df = df.dropna(subset=["price"])

# Handle missing values
num_cols_impute = ["year", "engine"]
cat_cols_impute = ["body", "fuel", "color", "registered"]

for c in num_cols_impute:
    df[c] = df[c].fillna(df[c].median())
for c in cat_cols_impute:
    df[c] = df[c].fillna(df[c].mode().iloc[0])

# Create binary target: HighPrice
median_price = df["price"].median()
df["HighPrice"] = (df["price"] >= median_price).astype(int)
print(f"Median price: {median_price:,.0f} PKR")
print(f"HighPrice distribution:\n{df['HighPrice'].value_counts()}")

# Drop price column
df = df.drop(columns=["price"])

# Select important features for the deployment model
feature_cols = ["year", "engine", "mileage", "transmission", "fuel", "body", "city"]
X = df[feature_cols].copy()
y = df["HighPrice"]

# Feature engineering
max_year = X["year"].max()
X["car_age"] = max_year - X["year"]
X["mileage_per_year"] = X["mileage"] / (X["car_age"] + 1)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=1
)
print(f"\nX_train: {X_train.shape} | X_test: {X_test.shape}")
print(f"y_train proportions: {y_train.value_counts(normalize=True).to_dict()}")

Median price: 2,700,000 PKR
HighPrice distribution:
HighPrice
1    31090
0    30740
Name: count, dtype: int64

X_train: (43281, 9) | X_test: (18549, 9)
y_train proportions: {1: 0.5028303412582888, 0: 0.49716965874171115}


In [6]:
# Build preprocessing pipeline
numerical_cols = ["year", "engine", "mileage", "car_age", "mileage_per_year"]
categorical_cols = ["transmission", "fuel", "body", "city"]

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numerical_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

# Build full pipeline with SVM (RBF kernel, best from Assignment 2)
svm_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("svm", SVC(kernel="rbf", probability=True, random_state=42))
])

# Train
print("Training SVM model...")
svm_pipeline.fit(X_train, y_train)

# Evaluate
y_pred = svm_pipeline.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"\nModel Accuracy: {accuracy:.4f}")
print(f"\nClassification Report:\n{classification_report(y_test, y_pred)}")

Training SVM model...

Model Accuracy: 0.9217

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.93      0.92      9222
           1       0.93      0.91      0.92      9327

    accuracy                           0.92     18549
   macro avg       0.92      0.92      0.92     18549
weighted avg       0.92      0.92      0.92     18549



In [7]:
# Save the trained model
joblib.dump(svm_pipeline, "pakwheels_svm_model.pkl")
print("Model saved as: pakwheels_svm_model.pkl")

# Verify saved model loads correctly
loaded_model = joblib.load("pakwheels_svm_model.pkl")
y_pred_verify = loaded_model.predict(X_test)
print(f"Verification accuracy: {accuracy_score(y_test, y_pred_verify):.4f}")
print("Model save and load verified successfully.")

Model saved as: pakwheels_svm_model.pkl
Verification accuracy: 0.9217
Model save and load verified successfully.
